In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [2]:
outcomes = pd.read_csv('outcomes_20260704_205922.csv')
decisions = pd.read_csv('decisions_20260704_205922.csv')
weights = pd.read_csv('weights_20260704_205922.csv')
zones = pd.read_csv('zones_20260704_205922.csv')


In [3]:
outcomes.head()

,day,stakeholder_id,name,requested,comfortable_threshold,critical_threshold,allocated,allocated_from_authority,allocated_from_trades,satisfaction,...,cooperated,objected,supply,total_allocated,peak_temp_c,ruling_text,cascade_increase,run_id,condition,seed
0,1,hospital,Hospital / Healthcare Services,150.0,132.0,97.5,150.0,150.0,0.0,1.000,...,False,False,1000.0,1000.0,34.0,"Given today’s severe shortage and high heat, w...",0.0,no_cascade_seed0,no_cascade,0
1,1,households,Households,400.0,312.0,220.0,365.0,365.0,0.0,0.912,...,False,False,1000.0,1000.0,34.0,"Given today’s severe shortage and high heat, w...",0.0,no_cascade_seed0,no_cascade,0
2,1,agriculture,Agriculture,250.0,180.0,120.0,208.1,208.1,0.0,0.832,...,False,False,1000.0,1000.0,34.0,"Given today’s severe shortage and high heat, w...",0.0,no_cascade_seed0,no_cascade,0
3,1,industry,Industry / Businesses,150.0,112.5,78.0,120.0,120.0,0.0,0.800,...,False,False,1000.0,1000.0,34.0,"Given today’s severe shortage and high heat, w...",0.0,no_cascade_seed0,no_cascade,0
4,1,energy_utility,Energy Utility,100.0,90.0,68.0,98.1,98.1,0.0,0.981,...,False,False,1000.0,1000.0,34.0,"Given today’s severe shortage and high heat, w...",0.0,no_cascade_seed0,no_cascade,0


In [4]:
decisions.head()

,day,stakeholder_id,name,event_type,round,move_type,units,trade_target,text,run_id,condition,seed,reasoning
0,1,hospital,Hospital / Healthcare Services,request,0,request,150.0,NaN,Our hospitals and clinics cannot safely reduce...,no_cascade_seed0,no_cascade,0,NaN
1,1,households,Households,request,0,request,400.0,NaN,Households need the full baseline today to pre...,no_cascade_seed0,no_cascade,0,NaN
2,1,agriculture,Agriculture,request,0,request,250.0,NaN,"Agriculture can absorb a single difficult day,...",no_cascade_seed0,no_cascade,0,NaN
3,1,industry,Industry / Businesses,request,0,request,150.0,NaN,Industry and businesses need our full operatio...,no_cascade_seed0,no_cascade,0,NaN
4,1,energy_utility,Energy Utility,request,0,request,100.0,NaN,We need our full cooling-water requirement tod...,no_cascade_seed0,no_cascade,0,NaN


In [5]:
weights.head()

,day,stakeholder_id,name,weight_before,weight_after,delta,reason,run_id,condition,seed
0,1,hospital,Hospital / Healthcare Services,5.0,5.0,0.0,no move today,no_cascade_seed0,no_cascade,0
1,1,households,Households,3.0,3.0,0.0,no move today,no_cascade_seed0,no_cascade,0
2,1,agriculture,Agriculture,2.0,2.0,0.0,no move today,no_cascade_seed0,no_cascade,0
3,1,industry,Industry / Businesses,1.0,1.0,0.0,no move today,no_cascade_seed0,no_cascade,0
4,1,energy_utility,Energy Utility,4.0,4.0,0.0,no move today,no_cascade_seed0,no_cascade,0


In [6]:
zones.head()

,day,stakeholder_id,name,zone,allocated,comfortable_threshold,critical_threshold,cascade_increase,surplus_above_critical,deficit_below_comfortable,run_id,condition,seed
0,1,hospital,Hospital / Healthcare Services,comfortable,150.0,132.0,97.5,0.0,52.5,0.0,no_cascade_seed0,no_cascade,0
1,1,households,Households,comfortable,365.0,312.0,220.0,0.0,145.0,0.0,no_cascade_seed0,no_cascade,0
2,1,agriculture,Agriculture,comfortable,208.1,180.0,120.0,0.0,88.1,0.0,no_cascade_seed0,no_cascade,0
3,1,industry,Industry / Businesses,comfortable,120.0,112.5,78.0,0.0,42.0,0.0,no_cascade_seed0,no_cascade,0
4,1,energy_utility,Energy Utility,comfortable,98.1,90.0,68.0,0.0,30.1,0.0,no_cascade_seed0,no_cascade,0


In [10]:
# Per-day aggregates per run
per_run_day = outcomes.groupby(['run_id','condition','seed','day']).agg(
    critical_failures  = ('critical_failure', 'sum'),
    mean_satisfaction  = ('satisfaction', 'mean'),
    min_satisfaction   = ('satisfaction', 'min'),
    fairness_gini      = ('satisfaction', lambda x: (
        np.abs(np.subtract.outer(x.values, x.values)).sum() / (2 * len(x) * x.sum())
        if x.sum() > 0 else 0
    )),
    severity           = ('severity_today', 'max'),
    rounds             = ('rounds_today', 'max'),
).reset_index()

# Per-run summary
rows = []
for (run_id, condition, seed), group in per_run_day.groupby(['run_id','condition','seed']):
    moves = decisions[
        (decisions['run_id']==run_id) &
        (decisions['event_type']=='move')
    ]
    n = len(moves)
    rows.append({
        'run_id': run_id, 'condition': condition, 'seed': seed,
        'cooperation_rate':                    round((moves['move_type'].isin(['concede','accept','propose_trade'])).sum() / n, 3) if n else float('nan'),
        'n_conflicts':                         int((moves['move_type']=='object').sum()),
        'n_compromises':                       int((moves['move_type']=='concede').sum()),
        'n_trades_proposed':                   int((moves['move_type']=='propose_trade').sum()),
        'n_reactive_cooperative':              int(moves['move_type'].isin(['concede','accept','propose_trade']).sum()),
        'n_critical_failures':                 int(group['critical_failures'].sum()),
        'mean_fairness_gini':                  round(group['fairness_gini'].mean(), 3),
        'mean_collective_welfare_utilitarian': round(group['mean_satisfaction'].mean(), 3),
        'mean_collective_welfare_rawlsian':    round(group['min_satisfaction'].mean(), 3),
    })

per_run_summary = pd.DataFrame(rows)

condition_order = ['no_cascade', 'cascade', 'deeper_cascade']
condition_labels = {
    'no_cascade': 'no-cascade condition',
    'cascade': 'cascade condition',
    'deeper_cascade': 'deeper-cascade condition',
}

print(
    per_run_summary.groupby('condition')[['cooperation_rate', 'n_critical_failures', 'mean_collective_welfare_rawlsian', 'n_conflicts', 'n_compromises', 'n_trades_proposed', 'mean_fairness_gini', 'mean_collective_welfare_utilitarian'
    ]]
    .agg(['mean', 'std'])
    .reindex(condition_order)
    .rename(index=condition_labels)
)

                         cooperation_rate           n_critical_failures  \
                                     mean       std                mean   
condition                                                                 
no-cascade condition             0.533333  0.028868           10.000000   
cascade condition                0.555333  0.025423           10.333333   
deeper-cascade condition         0.461333  0.009815           14.000000   

                                  mean_collective_welfare_rawlsian            \
                              std                             mean       std   
condition                                                                      
no-cascade condition      0.00000                            0.223  0.005196   
cascade condition         0.57735                            0.223  0.005196   
deeper-cascade condition  0.00000                            0.117  0.007000   

                         n_conflicts           n_compromises       \

In [8]:

metrics = [
    'cooperation_rate',
    'n_conflicts',
    'n_critical_failures',
    'mean_fairness_gini',
    'mean_collective_welfare_utilitarian',
    'mean_collective_welfare_rawlsian',
]

no_cascade = per_run_summary[per_run_summary['condition'] == 'no_cascade']
cascade    = per_run_summary[per_run_summary['condition'] == 'cascade']
deeper     = per_run_summary[per_run_summary['condition'] == 'deeper_cascade']

print('=== Kruskal-Wallis (exploratory, n=3 per group) ===\n')
significant = []
for metric in metrics:
    g1, g2, g3 = no_cascade[metric].values, cascade[metric].values, deeper[metric].values
    # Skip if all values identical across groups (test undefined)
    if len(set(np.concatenate([g1, g2, g3]))) == 1:
        print(f'{metric:45} all values identical — skipped')
        continue
    h, p = stats.kruskal(g1, g2, g3)
    sig = '***' if p < 0.05 else '(ns)'
    print(f'{metric:45} H={h:.3f}  p={p:.3f}  {sig}')
    if p < 0.05:
        significant.append(metric)

# Pairwise Mann-Whitney U only for significant metrics
if significant:
    print('\n=== Pairwise Mann-Whitney U (post-hoc, significant metrics only) ===\n')
    pairs = [
        ('no_cascade',     'cascade'),
        ('no_cascade',     'deeper_cascade'),
        ('cascade',        'deeper_cascade'),
    ]
    for metric in significant:
        print(f'{metric}')
        for a, b in pairs:
            g1 = per_run_summary[per_run_summary['condition']==a][metric].values
            g2 = per_run_summary[per_run_summary['condition']==b][metric].values
            if len(set(np.concatenate([g1, g2]))) == 1:
                print(f'  {a} vs {b}: identical — skipped')
                continue
            u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
            sig = '***' if p < 0.05 else '(ns)'
            print(f'  {a} vs {b}: U={u:.1f}  p={p:.3f}  {sig}')
        print()
else:
    print('\nNo metrics reached significance. Report descriptive patterns and note low power (n=3 per group).')

=== Kruskal-Wallis (exploratory, n=3 per group) ===

cooperation_rate                              H=5.843  p=0.054  (ns)
n_conflicts                                   H=5.843  p=0.054  (ns)
n_critical_failures                           H=7.000  p=0.030  ***
mean_fairness_gini                            H=7.261  p=0.027  ***
mean_collective_welfare_utilitarian           H=5.647  p=0.059  (ns)
mean_collective_welfare_rawlsian              H=5.945  p=0.051  (ns)

=== Pairwise Mann-Whitney U (post-hoc, significant metrics only) ===

n_critical_failures
  no_cascade vs cascade: U=3.0  p=0.505  (ns)
  no_cascade vs deeper_cascade: U=0.0  p=0.047  ***
  cascade vs deeper_cascade: U=0.0  p=0.059  (ns)

mean_fairness_gini
  no_cascade vs cascade: U=0.0  p=0.077  (ns)
  no_cascade vs deeper_cascade: U=0.0  p=0.077  (ns)
  cascade vs deeper_cascade: U=0.0  p=0.100  (ns)



n_critical_failures (H=7.000, p=0.030): the overall test is significant. Post-hoc: only no_cascade vs deeper_cascade is significant (p=0.047). No_cascade and cascade are indistinguishable (p=0.505), but deeper_cascade produces meaningfully more critical failures than no_cascade. Cascade consequences alone (same supply, just with cascade) did not change the number of critical failures, only the combination of cascades and deeper scarcity did.


mean_fairness_gini (H=7.261, p=0.027): overall significant but none of the pairwise comparisons survive. This is a classic case where the overall test detects something but with n=3 per group the post-hoc has no power to localise it. The direction from your descriptive table will tell you which condition was most unequal


cooperation_rate (p=0.054), n_conflicts (p=0.054), mean_collective_welfare_rawlsian (p=0.051): all just above 0.05. With n=3 this is expected, the effect would likely be significant with more seeds. Worth reporting the H statistic and noting marginal significance rather than treating these as null.


Core finding:
Scarcity depth matters for outcomes even when cooperation is zero. Cascade consequences alone (Condition 2 vs 1) did not significantly change critical failures or fairness, but deeper scarcity with cascades (Condition 3 vs 1) did. This means the cascade mechanism's effect is contingent on scarcity being severe enough that agents cannot absorb the threshold increases.
